# FoundML 3 — Week 2 Coding Assignment (Student Version)
## Ensemble Learning without Trees: Voting and Stacking (Breast Cancer, ROC AUC)

In this assignment you will demonstrate the benefit of **combining diverse learners** using:

1. **Single models** (Logistic Regression, SVM, k-NN)
2. **Soft Voting** (average predicted probabilities)
3. **Stacking** (a meta-learner learns how to combine base learners)

Dataset: **Breast Cancer Wisconsin (Diagnostic)** from scikit-learn (binary classification).  
Main metric: **ROC AUC**.

Difficulty level: **moderate**.

All numerical answers must be rounded to **2 decimals**.


## Rules
- Use the provided random seeds and hyperparameters.
- Do not change the grading variable names (Q1–Q4).
- Use the pipelines exactly as specified (especially StandardScaler for SVM/k-NN).


## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, RocCurveDisplay

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.ensemble import VotingClassifier, StackingClassifier

np.random.seed(0)


## 1. Load the dataset and create a train/test split

### Data

From the documentation: https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-dataset


"This is a copy of UCI ML Breast Cancer Wisconsin (Diagnostic) datasets. https://goo.gl/U2Uwz2

Features are computed from a digitized image of a fine needle aspirate (FNA) of a breast mass. They describe characteristics of the cell nuclei present in the image.

Separating plane described above was obtained using Multisurface Method-Tree (MSM-T) [K. P. Bennett, “Decision Tree Construction Via Linear Programming.” Proceedings of the 4th Midwest Artificial Intelligence and Cognitive Science Society, pp. 97-101, 1992], a classification method which uses linear programming to construct a decision tree. Relevant features were selected using an exhaustive search in the space of 1-4 features and 1-3 separating planes.

The actual linear program used to obtain the separating plane in the 3-dimensional space is that described in: [K. P. Bennett and O. L. Mangasarian: “Robust Linear Programming Discrimination of Two Linearly Inseparable Sets”, Optimization Methods and Software 1, 1992, 23-34]."

In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target

X.shape, y.shape


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=0,
    stratify=y
)

X_train.shape, X_test.shape


## 2. Evaluation helper (do not modify)

We compute:
- 5-fold stratified **CV ROC AUC** on the training set
- **Test ROC AUC** on the held-out test set

Note: All models must implement `predict_proba` for ROC AUC evaluation.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

def eval_auc(model, X_train, y_train, X_test, y_test):
    cv_auc = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc").mean()

    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, proba)

    return float(cv_auc), float(test_auc)


## 3. Single models (baselines)

Build the following three models.

### 3.1 Logistic Regression
Use:
- `LogisticRegression(max_iter=1000, random_state=0)`

### 3.2 SVM (RBF kernel)
Use a pipeline:
- `StandardScaler()`
- `SVC(C=3.0, gamma="scale", probability=True, random_state=0)`

### 3.3 k-NN
Use a pipeline:
- `StandardScaler()`
- `KNeighborsClassifier(n_neighbors=15)`

**Task:** Create the three models and compute their (CV AUC, Test AUC).
Store results in:
- `lr_cv_auc`, `lr_test_auc`
- `svm_cv_auc`, `svm_test_auc`
- `knn_cv_auc`, `knn_test_auc`


In [ ]:
# Write here your own code.
# Requirements:
# - define lr, svm, knn (with pipelines as specified)
# - evaluate each using eval_auc
# - define the 6 variables:
#     lr_cv_auc, lr_test_auc
#     svm_cv_auc, svm_test_auc
#     knn_cv_auc, knn_test_auc
#
# YOUR CODE HERE


# SANITY CHECK (do not modify)

In [ ]:
# SANITY CHECK (do not modify)
required = ["lr_test_auc","svm_test_auc","knn_test_auc","lr_cv_auc","svm_cv_auc","knn_cv_auc"]
for r in required:
    assert r in globals(), f"{r} not defined"
for v in [lr_test_auc, svm_test_auc, knn_test_auc]:
    assert 0.5 <= v <= 1.0, "Test AUC should be in [0.5, 1.0]"
print("Sanity check passed: single-model AUCs look valid.")


### **Question 1**
Which **single model** achieved the highest **test ROC AUC**?

Set `Q1_best_single_model` to exactly one of:
- `"lr"`
- `"svm"`
- `"knn"`


In [ ]:
# ANSWER Q1
# Write your own code:
# - compare lr_test_auc, svm_test_auc, knn_test_auc
# - set Q1_best_single_model
#
# YOUR CODE HERE


## 4. Soft Voting Ensemble

Create a **soft voting** ensemble of the three base models (LR, SVM, kNN).

Use:
- `VotingClassifier(..., voting="soft")`

**Task:** Define the voting model and compute:
- `vote_cv_auc`, `vote_test_auc` using eval_auc.


In [ ]:
# Write here your own code.
# Requirements:
# - define voting (soft voting) using the three models (lr, svm, knn)
# - compute vote_cv_auc, vote_test_auc using eval_auc
#
# YOUR CODE HERE


### **Question 2**
What is the **test ROC AUC** of the **soft voting** ensemble? (rounded to 2 decimals)

Define:
- `Q2_vote_test_auc`


In [ ]:
# ANSWER Q2
# YOUR CODE HERE


## 5. Stacking Ensemble

Create a stacking model using the same base learners, and a logistic regression meta-learner.

Use:
- `final_estimator = LogisticRegression(max_iter=1000, random_state=0)`
- `passthrough = False`
- `stack_method = "predict_proba"`

**Task:** Define the stacking model and compute:
- `stack_cv_auc`, `stack_test_auc` using eval_auc.


In [ ]:
# Write here your own code.
# Requirements:
# - define stacking using lr, svm, knn as base estimators
# - final_estimator is LogisticRegression(max_iter=1000, random_state=0)
# - stack_method="predict_proba"
# - passthrough=False
# - compute stack_cv_auc, stack_test_auc
#
# YOUR CODE HERE


### **Question 3**
What is the **test ROC AUC** of the **stacking** ensemble? (rounded to 2 decimals)

Define:
- `Q3_stack_test_auc`


In [ ]:
# ANSWER Q3
# YOUR CODE HERE


### **Question 4**
Which of these achieved the highest **test ROC AUC**?

Choose one of:
- `"best_single"` (the best of lr/svm/knn)
- `"voting"`
- `"stacking"`

Define:
- `Q4_best_overall`


In [ ]:
# ANSWER Q4
# Write your own code.
# Requirements:
# - compare best single test AUC vs vote_test_auc vs stack_test_auc
# - define Q4_best_overall
#
# YOUR CODE HERE


# SANITY CHECK (do not modify)

In [ ]:
# SANITY CHECK (do not modify)
assert 'Q4_best_overall' in globals(), "Q4_best_overall not defined"
assert Q4_best_overall in ["best_single","voting","stacking"], "Invalid choice for Q4_best_overall"
print("Sanity check passed: Q4_best_overall =", Q4_best_overall)


## 6. Visualization (not graded)

1. Plot ROC curves on the **test set** for:
   - best single model
   - soft voting
   - stacking
2. Make a bar chart comparing **CV AUC** across:
   - LR, SVM, kNN, Voting, Stacking


In [ ]:
import matplotlib.pyplot as plt

# Write here your own code.
# Requirements:
# - Fit the models you need and plot ROC curves using RocCurveDisplay.from_estimator
# - Bar chart for CV AUC values
#
# YOUR CODE HERE


## Submission checklist

Your notebook must define:
- `Q1_best_single_model`
- `Q2_vote_test_auc`
- `Q3_stack_test_auc`
- `Q4_best_overall`
